In [1]:
import numpy as np, pandas as pd
from scipy import stats
seeds = [110, 111, 112, 113, 114]
rows = []
for s in seeds:
    r = pd.read_csv(f"../outputs/tables/extra_metrics_seed{s}.csv")
    rows.append(dict(seed=s, hospitals=len(r), nonlocal_per_hospital_alpha=int((r.sel_alpha005 > 0).sum()),
                     nonlocal_simultaneous=int((r.sel_bonferroni > 0).sum()),
                     eo_gap_local=r.eo_local.mean(), eo_gap_pooled=r.eo_pooled.mean(), eo_gap_certified=r.eo_certified.mean(),
                     median_loss_diff_pooled=r.diff_pooled.median(), wilcoxon_p_pooled=stats.wilcoxon(r.diff_pooled).pvalue,
                     median_loss_diff_certified=r.diff_certified.median(),
                     wilcoxon_p_certified=stats.wilcoxon(r.diff_certified).pvalue if (r.diff_certified != 0).any() else 1.0))
T = pd.DataFrame(rows); T.loc["mean"] = T.mean(numeric_only=True)
T.round(5).to_csv("../outputs/tables/T_extra_metrics.csv"); print(T.round(5).to_string())

comm = pd.DataFrame([
    ("FedAvg / FedProx, 30 rounds (logistic regression, 53 coefficients, 8-byte floats, up and down)", 25440, 848),
    ("Fine-tuning (one download of the pooled model)", 424, 424),
    ("Certificate, run at the hospital on its own certification split (no extra communication beyond the model download)", 424, 424),
    ("Certificate, central version (2 scalars per candidate and group; 4 candidates, 8 groups)", 512, 512)],
    columns=["method", "bytes_per_hospital_total", "bytes_per_round_or_message"])
comm.to_csv("../outputs/tables/T_communication.csv", index=False); print(); print(comm.to_string(index=False))

ledger = pd.DataFrame([
 ("0", "Pilot: tolerance grid, clipping, calibration power checks", "Exploratory; not reported as results"),
 ("100", "First loss-only run of the final code", "Exploratory (amendment 2)"),
 ("101-104", "Loss-only harm study and baselines (pooled, blends, certified, FedAvg, FedProx, fine-tuning, recalibration)", "Original confirmatory (amendments 2, 3)"),
 ("105-109", "Certified fine-tuned variant and baseline table (loss only)", "Confirmatory for amendment 4"),
 ("110-114", "Full certificate (loss and calibration) and all later analyses on the same splits: subgroup, intersectional, APR, temporal, size cutoffs, bound choice, extra metrics, participation, Ditto and q-FFL, model class (MLP 111-114), leave-one-hospital-out", "Headline confirmatory (amendments 5-8, 10, 11, 13-18); same splits across analyses, so not independent"),
 ("200-204", "ICU 2012 stress test and ICU diagnostic", "Amendments 12 and 19; resamples splits of the same 12,000 stays"),
 ("simulation 0-199", "Simulation with the actual certificate", "Amendment 9")],
 columns=["seeds", "used for", "status"])
ledger.to_csv("../outputs/tables/T4c_seed_ledger.csv", index=False)
print(); print(ledger.to_string(index=False))

       seed  hospitals  nonlocal_per_hospital_alpha  nonlocal_simultaneous  eo_gap_local  eo_gap_pooled  eo_gap_certified  median_loss_diff_pooled  wilcoxon_p_pooled  median_loss_diff_certified  wilcoxon_p_certified
0     110.0      126.0                         57.0                    6.0       0.11192        0.10105           0.10668                  0.00209                0.0                         0.0               0.06024
1     111.0      126.0                         56.0                    6.0       0.11201        0.10551           0.10897                  0.00171                0.0                         0.0               0.32765
2     112.0      126.0                         55.0                    5.0       0.11229        0.10910           0.11345                  0.00184                0.0                         0.0               0.09056
3     113.0      126.0                         56.0                    6.0       0.11575        0.10017           0.11063               

In [2]:
import shutil, os, pandas as pd
F, T = "../outputs/figures/", "../outputs/tables/"
figs = [("fig1_design", "design"), ("fig2_cohort_flow", "cohort_flow"), ("fig3_heterogeneity", "heterogeneity"),
 ("fig1_harm_by_candidate", "harm_by_candidate"), ("fig2_size_quartiles", "size_quartiles"), ("fig6_forest", "forest"),
 ("fig3_worst_regret", "worst_regret"), ("fig14_baselines", "baselines"), ("fig24_fair_baselines", "fair_baselines"),
 ("fig7_subgroup_calibration", "subgroup_calibration"), ("fig8_reliability", "reliability"), ("fig19_intersectional", "intersectional"),
 ("figA_clipping_ablation", "clipping_ablation"), ("fig11_chosen_blend", "chosen_blend"), ("fig12_price_of_no_harm", "price_of_no_harm"),
 ("fig18_certifiable_size", "certifiable_size"), ("fig22_size_cutoffs", "size_cutoffs"), ("fig9_phase_map", "phase_map"),
 ("fig10_simulation_certified", "simulation_certified"), ("fig23_participation", "participation"), ("fig16_temporal", "temporal"),
 ("fig15_model_class", "model_class"), ("fig20_apr_sensitivity", "apr_sensitivity"), ("fig21_loho", "loho"), ("fig17_icu2012", "icu2012")]
tabs = [("T1_cohort", "cohort"), ("T2_hospital_characteristics", "hospital_characteristics"), ("T3_features", "features"),
 ("T4a_prereg_settings", "prereg_settings"), ("T4b_amendments", "amendments"), ("T4c_seed_ledger", "seed_ledger"), ("T5_power", "power"),
 ("T11b_certificate_seeds110-114", "main_certificate_results"), ("T10b_certificate_grid", "certificate_grid"),
 ("T11_baselines_seeds105-109", "baselines"), ("T_fair_baselines", "fair_baselines"), ("T8_subgroup_calibration", "subgroup_calibration"),
 ("T_intersectional", "intersectional"), ("T_size_cutoffs", "size_cutoffs"), ("T_bound_choice", "bound_choice"),
 ("T22_min_certified_size", "min_certified_size"), ("T_participation", "participation"), ("T_extra_metrics", "extra_metrics"),
 ("T_communication", "communication"), ("T15_simulation_grid", "simulation_grid"), ("T17_temporal", "temporal"),
 ("T14_model_class", "model_class"), ("T19_apr_sensitivity", "apr_sensitivity"), ("T_loho", "loho"),
 ("T18_icu2012", "icu2012"), ("T18b_icu2012_by_unit", "icu2012_by_unit"), ("T_icu_diagnostic", "icu_diagnostic"),
 ("F12_price_of_no_harm_grid", "price_of_no_harm_grid"), ("T21_software_settings", "software_settings")]
os.makedirs("../paper/assets/figures", exist_ok=True); os.makedirs("../paper/assets/tables", exist_ok=True)
man, missing = [], []
for i, (src, nm) in enumerate(figs, 1):
    for ext in ("png", "pdf"):
        (shutil.copy(f"{F}{src}.{ext}", f"../paper/assets/figures/Fig{i:02d}_{nm}.{ext}") if os.path.exists(f"{F}{src}.{ext}") else missing.append(f"{src}.{ext}"))
    man.append(("figure", f"Fig{i:02d}", nm, src))
for i, (src, nm) in enumerate(tabs, 1):
    (shutil.copy(f"{T}{src}.csv", f"../paper/assets/tables/Table{i:02d}_{nm}.csv") if os.path.exists(f"{T}{src}.csv") else missing.append(f"{src}.csv"))
    man.append(("table", f"Table{i:02d}", nm, src))
pd.DataFrame(man, columns=["type", "label", "name", "source_file"]).to_csv("../paper/assets/MANIFEST.csv", index=False)
print(len(figs), "figures,", len(tabs), "tables; missing:", missing if missing else "none")

25 figures, 29 tables; missing: none


In [1]:
import pandas as pd
for s in range(200, 205):
    a = pd.read_csv(f"../outputs/tables/backup_icu/icu2012_seed{s}.csv"); b = pd.read_csv(f"../outputs/tables/icu2012_seed{s}.csv")
    pd.testing.assert_frame_equal(a, b, check_exact=False, rtol=1e-9, atol=1e-12)
print("ICU main results identical on all five seeds")

ICU main results identical on all five seeds


In [2]:
import numpy as np, pandas as pd
R = pd.concat([pd.read_csv(f"../outputs/tables/icu2012_seed{s}.csv") for s in range(200, 205)], ignore_index=True)
print("pooled mean loss difference:", round(R["pool_full__diff"].mean(), 4), "(earlier: -0.0244)")
print("pooled mean AUROC gain:", round((R["pool_full__auroc"] - R["local__auroc"]).mean(), 4), "(earlier: 0.0251)")
print("certified pooled non-local decisions at eps 0.02, delta 0.02:", int((R["cert_pool_e0.02_d0.02__blend"] > 0).sum()), "of 20 (earlier: 2)")
print("certified fine-tuned non-local decisions, all three pairs:",
      int(sum((R[f"cert_ft_e{e}_d{d}__blend"] > 0).sum() for e, d in [(0.005, 0.01), (0.01, 0.015), (0.02, 0.02)])), "of 60 (earlier: 0)")

pooled mean loss difference: -0.0244 (earlier: -0.0244)
pooled mean AUROC gain: 0.0251 (earlier: 0.0251)
certified pooled non-local decisions at eps 0.02, delta 0.02: 2 of 20 (earlier: 2)
certified fine-tuned non-local decisions, all three pairs: 0 of 60 (earlier: 0)


In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from noharm_fl.data.features import load
X, y, h, split, keep = load(110)
tr = split == "train"
m = LogisticRegression(C=1.0, max_iter=300).fit(X[tr], y[tr])
print("pooled mean training loss:", round(log_loss(y[tr], m.predict_proba(X[tr])[:, 1]), 5))

pooled mean training loss: 0.11712
